In [27]:
import pandas as pd
import json
import glob
import datetime
import PySimpleGUI as sg
import ast
import re


import subprocess
import sys
import os
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

## Update Log Files

In [28]:
def updateLogs(how=1):
    #  how    1 = update just this month (log.json.nn
    #         2 = update Last Months file (log_backup_year_mo.json
    #         3 = update all this years log files 
    pw=os.environ["OIT_PW"]
    #  compute previous month
    x = datetime.datetime.today() - datetime.timedelta(days=30)
    monthM = x.month  
    yearM = x.year
    
    year = datetime.datetime.today().year
    
    string = f'''sshpass -p {pw} sftp -o HostKeyAlgorithms=+ssh-rsa -o PubkeyAcceptedAlgorithms=+ssh-rsa  giddensm@165.127.62.8 << !'''

    if how == 1:  #  update this month
        string+= "\nmget /usr/local/cim/bic_etl/general/logs/log.json.[0-9]* logs"
    elif how == 2: 
        string+=f"\nmget /usr/local/cim/bic_etl/general/logs/log_backup_{yearM}_{monthM}*.json logs"
    elif how == 3:
        string+=f"\nmget /usr/local/cim/bic_etl/general/logs/log_backup_{year}*.json logs"
    
    string+="\n!'''"

    print(string)
    result=subprocess.run([string], shell=True, stdout=subprocess.PIPE)
    x = str(result.stdout)
    nfiles=0
    files=x.split("\\n")
    for line in files:
        print(line)
        if "fetch" in line.lower():
            nfiles+=1
    return nfiles,files
    

## Get Old Log Files

In [29]:
print (sys.platform)

linux


In [30]:
#! dir \\wsl.localhost\Ubuntu\home\joe\logs
latest = ""
latest_file=""
if sys.platform == "linux":
    path="logs"
else:
    path= "\\\\wsl.localhost\\Ubuntu\\home\\joe\\work\\Logs\\"
    
for x in os.listdir(path):
        if re.findall(".zip$",x):
            dat = re.findall("\d{4}_\d+",x)
            
            if str(dat[0]) > latest:
                latest=dat[0]
                latest_file=x
print(f"Latest File {latest_file}")


Latest File 


In [31]:
files2Process = []
for x in os.listdir(path):
        if re.findall("\d{4}_\d+.json$",x) or re.findall("^log.json\.\d+",x):
             files2Process.append(x)

In [ ]:
files2Process

In [33]:
nerr=0
nfail=0
nfile=0
stats={}
statsYrMo = {}
statsYrMoDy = {}
statsYrMoName = {}
statsNameYrMo = {}

fout = open("data/error-log.json","w")
bad = []
statsMetaDataYrMoDy={}
#statsYrMoDyDf = pd.DataFrame(columns=["Year","Month","Jan","Feb","Mar","Apr","May","Jun","Aug","Sep","Oct","Nov","Dec"])

for yr in range(2019,2024):
    statsYrMoDy[yr] = {}
    statsMetaDataYrMoDy[yr] = {}
    
    for mo in range(1,13):
        statsYrMoDy[yr][mo] = {}
        statsMetaDataYrMoDy[yr][mo] = {}        
        for dy in range(1,32):
            statsYrMoDy[yr][mo][dy]=0
            statsMetaDataYrMoDy[yr][mo][dy]={}
            

for file in files2Process:
    nfile+=1
    print(file)
    with open(f"{path}/{file}") as jsf:
       
        try:
            log=[]
            nline=0
            for line in jsf:
                try: 
       #         log.append(json.loads(line))
          #         fout.write(line)
                   log.append(ast.literal_eval(line))
                except:
                   bad.append(line)
                nline+=1
        
            for line in log:
                # if (nfile%5 ==0):
                #    print(line)
                if line["msg"].lower().find("error") >= 0:
                    nerr+=1
                 
     #               date_object = datetime.strptime(line['time'], '%Y-%m-%dT%h:%M:%s:%fZ').date()
    #                date_object = datetime.strptime(line['time'], '%Y-%m-%d').date()
                    yr = int( line['time'][0:4])
                    mo = int(line['time'][5:7])
                    dy = int(line['time'][8:10])
            
                    name = line['name']
                    if name == "Metadata Updater":
                         spl = line['msg'].split(" ")
                         ds=spl[3]
                         if ds in statsMetaDataYrMoDy[yr][mo][dy]:
                            statsMetaDataYrMoDy[yr][mo][dy][ds]+= 1
                         else:
                            statsMetaDataYrMoDy[yr][mo][dy][ds]= 1
                        
                    fout.write(f"{str(line)}\n")
                    
                    
                    if yr in statsYrMo and mo in statsYrMo[yr]:
                         statsYrMo[yr][mo]+=1
                    elif yr not in statsYrMo:
                        statsYrMo[yr] = {}
                        statsYrMo[yr][mo]=1
                    elif mo not in statsYrMo[yr]:
                        statsYrMo[yr][mo]=1
                       
                    if yr not in statsYrMoDy:
                            statsYrMoDy[yr] = {}
                    elif mo not in statsYrMoDy[yr]:
                            statsYrMoDy[yr][mo] = {}
                    elif dy not in statsYrMoDy[yr][mo]:
                            statsYrMoDy[yr][mo][dy] = 1
                    else:
                            statsYrMoDy[yr][mo][dy]+= 1
                        
                      
                    if yr in statsYrMoName and mo in statsYrMoName[yr]:
                        if name in statsYrMoName[yr][mo]:
                            statsYrMoName[yr][mo][name]+=1
                        else:
                            statsYrMoName[yr][mo][name]=1
                    elif yr not in statsYrMoName:
                        statsYrMoName[yr] = {}
                        statsYrMoName[yr][mo] = {}
                        statsYrMoName[yr][mo][name]=1
                    elif mo not in statsYrMoName[yr]:
                        statsYrMoName[yr][mo] = {}
                        statsYrMoName[yr][mo][name]=1
                    elif name not in statsYrMoName[yr][mo]:
                        statsYrMoName[yr][mo][name]=1
                
                    if name in statsNameYrMo and yr in statsNameYrMo[name]:
                        if mo in statsNameYrMo[name][yr]:
                            statsNameYrMo[name][yr][mo]["count"]+=1
                            statsNameYrMo[name][yr][mo]["errors"].append(line['msg'])
                        else:
                            statsNameYrMo[name][yr][mo] = {}
                            statsNameYrMo[name][yr][mo]["count"]=1
                            statsNameYrMo[name][yr][mo]["errors"]= []
                            statsNameYrMo[name][yr][mo]["errors"].append(line['msg'])
                                  
                    elif name not in statsNameYrMo:
                        statsNameYrMo[name] = {}
                        statsNameYrMo[name][yr] = {}
                        statsNameYrMo[name][yr][mo] = {}
                        statsNameYrMo[name][yr][mo]["count"]=1
                        statsNameYrMo[name][yr][mo]["errors"]= []
                        statsNameYrMo[name][yr][mo]["errors"].append(line['msg'])
                    elif yr not in statsNameYrMo[name]:
                        statsNameYrMo[name][yr] = {}
                        statsNameYrMo[name][yr][mo] = {}
                        statsNameYrMo[name][yr][mo]["count"]=1
                        statsNameYrMo[name][yr][mo]["errors"]= []
                        statsNameYrMo[name][yr][mo]["errors"].append(line['msg'])
                    elif mo not in statsNameYrMo[name][yr]:
                        statsNameYrMo[name][yr][mo] = {}
                        statsNameYrMo[name][yr][mo]["count"]=1
                        statsNameYrMo[name][yr][mo]["errors"]= []
                        statsNameYrMo[name][yr][mo]["errors"].append(line['msg'])
                        
                        
                        
                    if line['name'] in stats:
                        stats[line['name']]+=1
                    else:
                        stats[line['name']]=1
        except Exception as e: print(nfile,nline,file,e)
           
            #            log.append(json.loads(line))
   
for yr in sorted( statsYrMo.keys()):
#       for mo in sorted(statsYrMo[yr].keys()):
        for mo in range(1,13):
            if mo not in statsYrMo[yr]:
                statsYrMo[yr][mo] = 0 

statsYrMoValues = []
for yr in sorted(statsYrMo.keys()):
     vals=[]
     vals.append(yr)
     for k,val in sorted(statsYrMo[yr].items()):
        vals.append(val)
     statsYrMoValues.append(vals)
     print(yr,vals)            
print(statsYrMoValues)
#        
print(nerr)  
print(nfail)

log_backup_2023_2.json
log_backup_2021_11.json
log_backup_2023_6.json
log_backup_2022_3.json
log_backup_2021_2.json
log_backup_2020_6.json
log_backup_2020_4.json
log_backup_2023_0.json
log_backup_2020_7.json
log_backup_2021_10.json
log_backup_2021_0.json
log_backup_2020_11.json
log_backup_2022_9.json
log_backup_2022_4.json
log_backup_2021_7.json
log_backup_2019_10.json
log_backup_2023_4.json
log_backup_2022_6.json
log_backup_2022_7.json
log_backup_2023_5.json
log_backup_2020_8.json
log_backup_2020_3.json
log_backup_2022_0.json
log_backup_2022_5.json
log.json.0
log.json.3
log_backup_2020_0.json
log_backup_2020_5.json
log_backup_2022_1.json
log_backup_2021_4.json
log_backup_2019_9.json
log_backup_2022_11.json
log_backup_2022_10.json
log.json.1
log_backup_2021_8.json
log_backup_2019_11.json
log_backup_2023_1.json
log_backup_2020_2.json
log_backup_2021_6.json
log_backup_2020_9.json
log_backup_2023_3.json
log_backup_2022_2.json
log_backup_2022_8.json
log_backup_2021_9.json
log_backup_2021_1

In [ ]:
nrows=0
for yr in sorted(statsYrMoDy.keys()):
     for mo in sorted(statsYrMoDy[yr].keys()):
            nrows+=1  
            dfTmp = pd.DataFrame({"Year":yr,"Month":mo} | {key:[val] for key,val in sorted(statsYrMoDy[yr][mo].items())})
            if nrows > 1:
                statsYrMoDyDf = pd.concat([statsYrMoDyDf,dfTmp])
            else:
                statsYrMoDyDf=dfTmp
                                      

In [ ]:
display(statsYrMoDyDf)

In [ ]:
## OD OLD OLD OLD OLD
nerr=0
nfail=0
nfile=0
stats={}
statsYrMo = {}
statsYrMoName = {}
statsNameYrMo = {}

fout = open("data/error-log.json","w")
bad = []
for file in glob.glob("attachments/*"):
    nfile+=1
    with open(file) as jsf:
       
        try:
            log=[]
            nline=0
            for line in jsf:
                try: 
       #         log.append(json.loads(line))
          #         fout.write(line)
                   log.append(ast.literal_eval(line))
                except:
                   bad.append(line)
                nline+=1
        
            for line in log:
                # if (nfile%5 ==0):
                #    print(line)
                if line["msg"].lower().find("error") >= 0:
                    nerr+=1
                 
     #               date_object = datetime.strptime(line['time'], '%Y-%m-%dT%h:%M:%s:%fZ').date()
    #                date_object = datetime.strptime(line['time'], '%Y-%m-%d').date()
                    yr = int( line['time'][0:4])
                    mo = int(line['time'][5:7])
                    name = line['name']
                    fout.write(f"{str(line)}\n")
                    if yr in statsYrMo and mo in statsYrMo[yr]:
                         statsYrMo[yr][mo]+=1
                    elif yr not in statsYrMo:
                        statsYrMo[yr] = {}
                        statsYrMo[yr][mo]=1
                    elif mo not in statsYrMo[yr]:
                        statsYrMo[yr][mo]=1
                       
                        
                    if yr in statsYrMoName and mo in statsYrMoName[yr]:
                        if name in statsYrMoName[yr][mo]:
                            statsYrMoName[yr][mo][name]+=1
                        else:
                            statsYrMoName[yr][mo][name]=1
                    elif yr not in statsYrMoName:
                        statsYrMoName[yr] = {}
                        statsYrMoName[yr][mo] = {}
                        statsYrMoName[yr][mo][name]=1
                    elif mo not in statsYrMoName[yr]:
                        statsYrMoName[yr][mo] = {}
                        statsYrMoName[yr][mo][name]=1
                    elif name not in statsYrMoName[yr][mo]:
                        statsYrMoName[yr][mo][name]=1
                
                    if name in statsNameYrMo and yr in statsNameYrMo[name]:
                        if mo in statsNameYrMo[name][yr]:
                            statsNameYrMo[name][yr][mo]["count"]+=1
                            statsNameYrMo[name][yr][mo]["errors"].append(line['msg'])
                        else:
                            statsNameYrMo[name][yr][mo] = {}
                            statsNameYrMo[name][yr][mo]["count"]=1
                            statsNameYrMo[name][yr][mo]["errors"]= []
                            statsNameYrMo[name][yr][mo]["errors"].append(line['msg'])
                                  
                    elif name not in statsNameYrMo:
                        statsNameYrMo[name] = {}
                        statsNameYrMo[name][yr] = {}
                        statsNameYrMo[name][yr][mo] = {}
                        statsNameYrMo[name][yr][mo]["count"]=1
                        statsNameYrMo[name][yr][mo]["errors"]= []
                        statsNameYrMo[name][yr][mo]["errors"].append(line['msg'])
                    elif yr not in statsNameYrMo[name]:
                        statsNameYrMo[name][yr] = {}
                        statsNameYrMo[name][yr][mo] = {}
                        statsNameYrMo[name][yr][mo]["count"]=1
                        statsNameYrMo[name][yr][mo]["errors"]= []
                        statsNameYrMo[name][yr][mo]["errors"].append(line['msg'])
                    elif mo not in statsNameYrMo[name][yr]:
                        statsNameYrMo[name][yr][mo] = {}
                        statsNameYrMo[name][yr][mo]["count"]=1
                        statsNameYrMo[name][yr][mo]["errors"]= []
                        statsNameYrMo[name][yr][mo]["errors"].append(line['msg'])
                        
                        
                        
                    if line['name'] in stats:
                        stats[line['name']]+=1
                    else:
                        stats[line['name']]=1
        except Exception as e: print(nfile,nline,file,e)
           
            #            log.append(json.loads(line))
   
for yr in sorted( statsYrMo.keys()):
#       for mo in sorted(statsYrMo[yr].keys()):
        for mo in range(1,13):
            if mo not in statsYrMo[yr]:
                statsYrMo[yr][mo] = 0 

statsYrMoValues = []
for yr in sorted(statsYrMo.keys()):
     vals=[]
     vals.append(yr)
     for k,val in sorted(statsYrMo[yr].items()):
        vals.append(val)
     statsYrMoValues.append(vals)
     print(yr,vals)            
print(statsYrMoValues)
#        
print(nerr)  
print(nfail)

In [34]:
for yr in sorted( statsYrMo.keys()):
#       for mo in sorted(statsYrMo[yr].keys()):
        for mo in range(1,13):
            if mo not in statsYrMo[yr]:
                statsYrMo[yr][mo] = 0 
       #     print(yr,mo,statsYrMo[yr][mo])

In [35]:
statsYrMoValues = []
for yr in sorted(statsYrMo.keys()):
     vals=[]
     vals.append(yr)
     for k,val in sorted(statsYrMo[yr].items()):
        vals.append(val)
     statsYrMoValues.append(vals)
     print(yr,vals)            
print(statsYrMoValues)

2019 [2019, 0, 0, 0, 0, 0, 0, 0, 0, 418, 82, 3, 0]
2020 [2020, 3, 7, 58, 119, 156, 138, 108, 255, 56, 404, 139, 0]
2021 [2021, 270, 416, 194, 15, 24, 601, 834, 855, 1009, 810, 780, 0]
2022 [2022, 1038, 808, 968, 1027, 1358, 1238, 1707, 530, 147, 148, 179, 0]
2023 [2023, 334, 192, 11032, 200, 276, 369, 39, 0, 0, 0, 0, 0]
[[2019, 0, 0, 0, 0, 0, 0, 0, 0, 418, 82, 3, 0], [2020, 3, 7, 58, 119, 156, 138, 108, 255, 56, 404, 139, 0], [2021, 270, 416, 194, 15, 24, 601, 834, 855, 1009, 810, 780, 0], [2022, 1038, 808, 968, 1027, 1358, 1238, 1707, 530, 147, 148, 179, 0], [2023, 334, 192, 11032, 200, 276, 369, 39, 0, 0, 0, 0, 0]]


In [ ]:
for yr in sorted(statsYrMoName.keys()):
    for mo in sorted(statsYrMoName[yr]):
        print(yr,mo,statsYrMo[yr][mo])
        for k, v in sorted(statsYrMoName[yr][mo].items(), key=lambda item: item[1],reverse=True):
              print(v,k)

In [36]:
for name in sorted(statsNameYrMo.keys()):
    for yr in sorted(statsNameYrMo[name]):
            for mo in range(1,13):
                  # if mo not in statsNameYrMo[name][yr]:
                  #       statsNameYrMo[name][yr][mo]=0
            for k, v in sorted(statsNameYrMo[name][yr].items(), key=lambda item: item[0],reverse=False):
                 print(name,yr,k,v)
nameYrMoNames = sorted(statsNameYrMo.keys())

IndentationError: expected an indented block after 'for' statement on line 3 (2759916537.py, line 6)

## GUI


In [37]:
def getYrMoNamebyMonth(year,month):
    vals = []
    counts=0
    for error, count in sorted(statsYrMoName[year][month].items(), key=lambda item: item[1],reverse=True):
            val = []
            val.append(year)
            val.append(month)
            val.append(count)
            counts+=count
            val.append(error)
            vals.append(val)
    val=[]
    val.append("Total")
    val.append(month)
    val.append(counts)
    val.append("---")
    vals.append(val)
    return vals

def getErrors(name):
    vals = []
    print("ERROR ",name)
    for yr in sorted(statsNameYrMo[name]):
           #   for mo in range(1,13):
                  # if mo not in statsNameYrMo[name][yr]:
                  #       statsNameYrMo[name][yr][mo]=0
            for mo, moInfo in sorted(statsNameYrMo[name][yr].items(), key=lambda item: item[0],reverse=False):
                 val = []
                 val.append(yr)
                 val.append(mo)
                 val.append(moInfo['count'])
                 val.append(name)
                 vals.append(val)
    return vals

def getNameErrors(yr,mo,name):
    vals = []
    for line in statsNameYrMo[name][yr][mo]['errors']:
        vals.append(line)
    return '\n'.join(vals)

nameYrMoNames = sorted(statsNameYrMo.keys())

In [38]:
months = ["","Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
header_list = ["Year","Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
###  PRIMARY LAYOUT

layout = [
         [sg.Text("BIC Update Errors")],
         [sg.Button('Quit')],
         [sg.Multiline("Update Info\nBe Certain to Connection via VPN before updating",s=(50,5),key="-UPDATES-")],
         [sg.Button("Update Logs This Month"),
           sg.Button("Update Logs Last Month"),
           sg.Button("Update Logs This Year")],

         
         [sg.Combo(nameYrMoNames,s=[65,10],expand_y=True,default_value=nameYrMoNames[0],key="-ERRORS-",enable_events=True)],
          [sg.Table(values=statsYrMoValues, max_col_width=10,
               background_color='green',
               auto_size_columns=True,enable_events=True,
               justification='center',alternating_row_color='brown',
               key='-TABLE-', headings = header_list)]
   ]
# Create the Window
sg.theme('Dark Green 5')
window = sg.Window('Output', layout,finalize=True,resizable=True)


#window2.move(window.current_location()[0]+600, window.current_location()[1])
table = window['-TABLE-']
table.bind('<Button-1>', "Click")
###  PRIMARY LOOP
while True:
  
    event, values = window.read()

#    window, event, values = sg.read_all_windows()
    if event == sg.WIN_CLOSED or event == 'Quit':
        window.close()
#        sys.exit(1)
        what = "QUIT"
        break
    elif event == '-ERRORS-':
        nameYrMoErrors = getErrors(values["-ERRORS-"])
        if name == "Metadata Updater":
            print(nameYrMoErrors)
            
        layout3 = [    
               [sg.Text(f"Summary of  {values['-ERRORS-']}")],
               [sg.Table(values=nameYrMoErrors, max_col_width=30,
               background_color='green',
               auto_size_columns=True,enable_events=True,
               justification='center',alternating_row_color='brown',
               key='-TABLE3-', headings = ["Year","Month","Count","Error"])]
        ]
        window3 = sg.Window(f"Errors for {months[month]}, {year}", layout3,finalize=True,resizable=True)
    elif event == '-TABLE-':
        pass
    elif event == "Update Logs This Month":
        nfiles,files = updateLogs(1)
        string=f"{nfiles} Downloaded\n"
        for file in files:
            string+=f"{file}\n"
        windows["-UPDATES-"].update(string)
    elif event == '-TABLE-Click':
        e = table.user_bind_event
        region = table.Widget.identify('region', e.x, e.y)
        if region == 'heading':
            row = 0
        elif region == 'cell':
            row = int(table.Widget.identify_row(e.y))
        elif region == 'separator':
            continue
        else:
            continue
        column = int(table.Widget.identify_column(e.x)[1:])
        year = statsYrMoValues[row-1][0]
        month = column-1
        yrMoNameValues = getYrMoNamebyMonth(year,month)
        layout2 = [    
               [sg.Text(f"Errors for {months[month]}, {year}")],
               [sg.Button('Close')],
               [sg.Table(values=yrMoNameValues, max_col_width=30,
               background_color='green',
               auto_size_columns=True,enable_events=True,
               justification='center',alternating_row_color='brown',
               key='-TABLE2-', headings = ["Year","Month","Count","Error"])]
        ]
        window2 = sg.Window(f"All Errors for {months[month]}, {year}", layout2,finalize=True,resizable=True)
        table2 = window2['-TABLE2-']
        table2.bind('<Button-1>', "Click")

        while True:

            event, values = window2.read()
    
        #    window, event, values = sg.read_all_windows()
            if event == sg.WIN_CLOSED or event == 'Close':
                window2.close()
        #        sys.exit(1)
                what = "QUIT"
                break
            elif event == '-TABLE2-':
                pass
            elif event == '-TABLE2-Click':
                e = table2.user_bind_event
                region = table2.Widget.identify('region', e.x, e.y)
                if region == 'heading':
                    row = 0
                elif region == 'cell':
                    row = int(table2.Widget.identify_row(e.y))
                elif region == 'separator':
                    continue
                else:
                    continue
                column = int(table2.Widget.identify_column(e.x)[1:])
                #year = statsYrMoValues[row-1][0]
                # month = column-1
                error = yrMoNameValues[row-1][3]
                nameErrorsbyYearMonth = getNameErrors(year,month,error)
         
                layout4 = [    
                       [sg.Text(f"Errors for {months[month]}, {year}")],
                       [sg.Button('Close')],
                       [sg.Multiline(nameErrorsbyYearMonth,size=[80,20],horizontal_scroll=True)]
                       
                ]  
                window4 = sg.Window(f"All Errors for {months[month]}, {year}", layout4,finalize=True,resizable=True)
                while True:

                    event, values = window4.read()
                #    window, event, values = sg.read_all_windows()
                    if event == sg.WIN_CLOSED or event == 'Close':
                        window4.close()
                #        sys.exit(1)
                        what = "QUIT"
                        break


sshpass -p s@AXuwr7 sftp -o HostKeyAlgorithms=+ssh-rsa -o PubkeyAcceptedAlgorithms=+ssh-rsa  giddensm@165.127.62.8 << !
mget /usr/local/cim/bic_etl/general/logs/log.json.[0-9]* logs
!'''


CentOS release 6.5 (Final)
Connected to 165.127.62.8.


b"sftp> mget /usr/local/cim/bic_etl/general/logs/log.json.[0-9]* logs
Fetching /usr/local/cim/bic_etl/general/logs/log.json.0 to logs/log.json.0
Fetching /usr/local/cim/bic_etl/general/logs/log.json.1 to logs/log.json.1
Fetching /usr/local/cim/bic_etl/general/logs/log.json.2 to logs/log.json.2
Fetching /usr/local/cim/bic_etl/general/logs/log.json.3 to logs/log.json.3
Fetching /usr/local/cim/bic_etl/general/logs/log.json.4 to logs/log.json.4
sftp> !'''
"


Unterminated quoted argument


NameError: name 'windows' is not defined